In [0]:
import os
import sys
project_path = os.path.join(os.getcwd(),'..','..','..','..')
sys.path.append(project_path)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from src.spotifyproject_etl.utils.utils import reusable
df_object = reusable()

#### DimArtist
#### Load -> transform -> dump

In [0]:
df_artist = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimArtist/schema')\
            .load('abfss://bronze@spotifylakehouseproject.dfs.core.windows.net/rawdata/DimArtist')

In [0]:
df_artist = df_object.dropColumn(df_artist,['_rescued_data'])
# df_artist = df_artist.dropDuplicates(['artist_id'])

In [0]:
df_artist.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimArtist/schema')\
        .trigger(once = True)\
            .option('path','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimArtist/data')\
                .toTable('silvecatalog.silverschema.DimArtist')

#### DimDate
#### Load -> transform -> dump

In [0]:
df_date = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimDate/schema')\
            .load('abfss://bronze@spotifylakehouseproject.dfs.core.windows.net/rawdata/DimDate')

In [0]:
df_date = df_object.dropColumn(df_date,['_rescued_data'])
df_date = df_date.dropDuplicates(['date_key'])

In [0]:
df_date.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimDate/schema')\
        .trigger(once = True)\
            .option('path','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimDate/data')\
                .toTable('silvecatalog.silverschema.DimDate')

#### DimTrack
#### Load -> transform -> dump

In [0]:
df_track = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimTrack/schema')\
            .load('abfss://bronze@spotifylakehouseproject.dfs.core.windows.net/rawdata/DimTrack')

In [0]:
#created the updated_flag column with short, medium and long track flags
df_track = df_track.withColumn('updated_flag',when(col('duration_sec') < 150, 'short_track')\
    .when(col('duration_sec')< 250, 'medium_track')\
        .otherwise('long_track'))
# transform the track_name
df_track = df_track.withColumn('track_name', regexp_replace(col('track_name'),'-',' '))
# remove the _rescued_data column
df_track = df_object.dropColumn(df_track,['_rescued_data'])
# remove the duplicate records from date_key column
df_track = df_track.dropDuplicates(['track_id'])

In [0]:
df_track.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimTrack/schema')\
        .trigger(once = True)\
            .option('path','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimTrack/data')\
                .toTable('silvecatalog.silverschema.DimTrack')

#### DimUser
#### Load -> transform -> dump

In [0]:
df_user = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimUser/schema')\
            .load('abfss://bronze@spotifylakehouseproject.dfs.core.windows.net/rawdata/DimUser')

In [0]:
df_user = df_object.dropColumn(df_user,['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])

In [0]:
df_user.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimUser/schema')\
        .trigger(once = True)\
            .option('path','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/DimUser/data')\
                .toTable('silvecatalog.silverschema.DimUser')

#### FactStream
#### Load -> transform -> dump

In [0]:
df_stream = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/FactStream/schema')\
            .load('abfss://bronze@spotifylakehouseproject.dfs.core.windows.net/rawdata/FactStream')

In [0]:
df_stream = df_object.dropColumn(df_stream,['_rescued_data'])
df_stream = df_stream.dropDuplicates(['stream_id'])

In [0]:
df_stream.writeStream.format('delta')\
    .option('checkpointLocation','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/FactStream/schema')\
        .trigger(once = True)\
            .option('path','abfss://silver@spotifylakehouseproject.dfs.core.windows.net/FactStream/data')\
                .toTable('silvecatalog.silverschema.FactStream')